# Camera snapshot — Tang Nano 4K

Takes a 160x120 grayscale picture with the Tang Nano 4K's OV2640 camera and shows it here.
Everything goes over the one USB cable (JTAG through the onboard debugger).

The overlay has three general-purpose IPs next to the base design (LED, buttons):
- `cam_i2c`: an I2C master (the camera's SCCB configuration bus)
- `cam_dvp`: the camera clock (XCLK) and the pixel front end
- `cam_capture`: a 20 KB frame buffer

`Camera` puts them together with the OV2640 driver. Before you start, see `02_led_tangnano4k.ipynb`
(Zadig, `python -m nanopynq doctor --board tangnano4k`).

## 1. Load the camera overlay

In [ ]:
from nanopynq import Overlay
from nanopynq.camera import Camera

ol = Overlay("camera", board="tangnano4k")
ol

## 2. Put the camera together and start it

`start()` turns XCLK on, checks the sensor's chip ID over SCCB and sends the OV2640 set-up (about 1 s).

In [ ]:
cam = Camera(ol.cam_i2c, ol.cam_capture, ol.cam_dvp)
cam.start()

## 3. Take a picture

The first picture after `start()` can be darker while the auto exposure settles.

In [ ]:
cam.snapshot()

In [ ]:
frame = cam.snapshot()
frame

## 4. A few in a row

About 0.6 s per picture: 0.4 s to read the 19.2 kB over JTAG, the rest waiting for the next frame.

In [ ]:
import time

from IPython.display import display

for _ in range(3):
    t0 = time.perf_counter()
    f = cam.snapshot()
    print(f"{time.perf_counter() - t0:.2f} s")
    display(f)

## 5. The registers underneath

`FRAMES` counts the sensor's frames; `STATUS` bit 1 is done, bit 2 overflow; `LENGTH` is the bytes of the last capture.

In [ ]:
print("frames", ol.cam_dvp.frames)
print(
    "capture status", ol.cam_capture.read(0x4), "length", ol.cam_capture.read(0x8), "capacity", ol.cam_capture.capacity
)

## 6. Close

In [ ]:
ol.close()